In [1]:
!pip install econml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 8.9 MB/s eta 0:00:00
  Attempting uninstall: shap
    Found existing installation: shap 0.51.0
    Uninstalling shap-0.51.0:
      Successfully uninstalled shap-0.51.0


In [3]:
import pandas as pd

# Load your CSV
df = pd.read_csv("final.csv")
df = df.drop(columns=['_id_x', '_id_y'])
# Convert datetime column
df['datetime'] = pd.to_datetime(df['datetime'])

# Filter only year 2025
df_2025 = df[
    (df['datetime'] >= '2025-01-01') &
    (df['datetime'] < '2026-01-01')
]

# Save to new CSV file
df_2025.to_csv("2025_load.csv", index=False)

print("Saved successfully as 2025_load.csv")

Saved successfully as 2025_load.csv


In [5]:
import pandas as pd

# Load CSV
df = pd.read_csv("final.csv")
df = df.drop(columns=['_id_x', '_id_y'])
# Convert datetime
df['datetime'] = pd.to_datetime(df['datetime'])

# Filter required range: Dec 2024 → Nov 2025
df = df[
    (df['datetime'] >= '2024-12-01') &
    (df['datetime'] < '2025-12-01')
]

# Extract year and month
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month

# ---- Create seasonal subsets ----

# Winter: Dec 2024 + Jan-Feb 2025
winter = df[
    ((df['year'] == 2024) & (df['month'] == 12)) |
    ((df['year'] == 2025) & (df['month'].isin([1, 2])))
]

# Summer: Mar-May 2025
summer = df[
    (df['year'] == 2025) & (df['month'].isin([3, 4, 5]))
]

# Monsoon: Jun-Sep 2025
monsoon = df[
    (df['year'] == 2025) & (df['month'].isin([6, 7, 8, 9]))
]

# Autumn: Oct-Nov 2025
autumn = df[
    (df['year'] == 2025) & (df['month'].isin([10, 11]))
]

# ---- Save separate CSV files ----
winter.to_csv("winter_2025.csv", index=False)
summer.to_csv("summer_2025.csv", index=False)
monsoon.to_csv("monsoon_2025.csv", index=False)
autumn.to_csv("autumn_2025.csv", index=False)

print("All seasonal CSV files saved successfully!")

All seasonal CSV files saved successfully!


In [18]:
import pandas as pd

# Load CSV
df1 = pd.read_csv("final.csv")
df1 = df1.drop(columns=['_id_x', '_id_y'])
# Convert datetime
df1['datetime'] = pd.to_datetime(df1['datetime'])

# Filter required range: Dec 2025 → Feb 2026
df1 = df1[
    (df1['datetime'] >= '2025-12-01') &
    (df1['datetime'] < '2026-03-01')
]

# Extract year and month
df1['year'] = df1['datetime'].dt.year
df1['month'] = df1['datetime'].dt.month

# Winter: Dec 2025 + Jan-Feb 2026
winter = df1[
    ((df1['year'] == 2025) & (df1['month'] == 12)) |
    ((df1['year'] == 2026) & (df1['month'].isin([1, 2])))
]
winter.to_csv("winter_2026.csv", index=False)
print("winter_2026.csv saved successfully!")

winter_2026.csv saved successfully!


In [6]:
import pandas as pd
from sklearn.linear_model import LinearRegression

# Function to train and print equation
def run_regression(file_name, season_name):
    df = pd.read_csv(file_name)

    # Features and target
    X = df[['temp', 'rh']]
    y = df['wdLoad']

    # Model
    model = LinearRegression()
    model.fit(X, y)

    # Coefficients
    intercept = model.intercept_
    coef_temp = model.coef_[0]
    coef_rh = model.coef_[1]

    print(f"\n--- {season_name} ---")
    print(f"Equation:")
    print(f"wdLoad = {intercept:.4f} + ({coef_temp:.4f} * temp) + ({coef_rh:.4f} * rh)")

    return model

# Run for each season
winter_model = run_regression("winter_2025.csv", "Winter")
summer_model = run_regression("summer_2025.csv", "Summer")
monsoon_model = run_regression("monsoon_2025.csv", "Monsoon")
autumn_model = run_regression("autumn_2025.csv", "Autumn")


--- Winter ---
Equation:
wdLoad = -117.8124 + (18.7679 * temp) + (-0.1326 * rh)

--- Summer ---
Equation:
wdLoad = -436.0413 + (27.9236 * temp) + (0.4927 * rh)

--- Monsoon ---
Equation:
wdLoad = -205.0712 + (25.1965 * temp) + (-0.6180 * rh)

--- Autumn ---
Equation:
wdLoad = -122.3786 + (19.2465 * temp) + (0.2856 * rh)


In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the winter_2026.csv file
df_winter_2026 = pd.read_csv("winter_2026.csv")

# Coefficients from the winter equation derived earlier (from cell czIO5NAoXPP2 or cAP8kozcXy7v)
# wdLoad = -117.8124 + (18.7679 * temp) + (-0.1326 * rh)
intercept = -117.8124
coef_temp = 18.7679
coef_rh = -0.1326

# Predict wdLoad using the winter equation
df_winter_2026['predicted_wdLoad'] = intercept + \
                                     (coef_temp * df_winter_2026['temp']) + \
                                     (coef_rh * df_winter_2026['rh'])

# Actual wdLoad values
y_true = df_winter_2026['wdLoad']

# Predicted wdLoad values
y_pred = df_winter_2026['predicted_wdLoad']

# Calculate evaluation metrics

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_true, y_pred)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

# Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    # Avoid division by zero
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape = mean_absolute_percentage_error(y_true, y_pred)

print("\n--- Model Performance for Winter 2026 using Winter 2025 Equation ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.4f}%")

# Display the first few rows with actual and predicted values
print("\nActual vs. Predicted wdLoad (first 5 rows):")
display(df_winter_2026[['datetime', 'wdLoad', 'predicted_wdLoad']].head())


--- Model Performance for Winter 2026 using Winter 2025 Equation ---
Mean Absolute Error (MAE): 63.8963
Root Mean Squared Error (RMSE): 77.8376
Mean Absolute Percentage Error (MAPE): 14.8982%

Actual vs. Predicted wdLoad (first 5 rows):


,datetime,wdLoad,predicted_wdLoad
0,2025-12-01 00:00:00+00:00,298.534462,272.608830
1,2025-12-01 00:15:00+00:00,304.524118,271.630655
2,2025-12-01 00:30:00+00:00,319.374861,270.785080
3,2025-12-01 00:45:00+00:00,329.508837,271.962155
4,2025-12-01 01:00:00+00:00,343.143157,271.116580


In [7]:
import pandas as pd
import numpy as np

def manual_mlr(file_name, season_name):
    df = pd.read_csv(file_name)

    # Extract variables
    temp = df['temp'].values
    rh = df['rh'].values
    y = df['wdLoad'].values

    n = len(df)

    # Construct X matrix: [1, temp, rh]
    X = np.column_stack((np.ones(n), temp, rh))

    # Apply formula: β = (XᵀX)^(-1) Xᵀy
    Xt = X.T
    beta = np.linalg.inv(Xt @ X) @ Xt @ y

    b0, b1, b2 = beta

    print(f"\n--- {season_name} ---")
    print(f"wdLoad = {b0:.4f} + ({b1:.4f})·temp + ({b2:.4f})·rh")

# Run for all seasons
manual_mlr("winter_2025.csv", "Winter")
manual_mlr("summer_2025.csv", "Summer")
manual_mlr("monsoon_2025.csv", "Monsoon")
manual_mlr("autumn_2025.csv", "Autumn")


--- Winter ---
wdLoad = -117.8124 + (18.7679)·temp + (-0.1326)·rh

--- Summer ---
wdLoad = -436.0413 + (27.9236)·temp + (0.4927)·rh

--- Monsoon ---
wdLoad = -205.0712 + (25.1965)·temp + (-0.6180)·rh

--- Autumn ---
wdLoad = -122.3786 + (19.2465)·temp + (0.2856)·rh


In [8]:
import pandas as pd

def compute_sums(file):
    df = pd.read_csv(file)

    x1 = df['temp']
    x2 = df['rh']
    y = df['wdLoad']

    n = len(df)

    sums = {
        "n": n,
        "sum_y": y.sum(),
        "sum_x1": x1.sum(),
        "sum_x2": x2.sum(),
        "sum_x1_sq": (x1**2).sum(),
        "sum_x2_sq": (x2**2).sum(),
        "sum_x1x2": (x1 * x2).sum(),
        "sum_x1y": (x1 * y).sum(),
        "sum_x2y": (x2 * y).sum()
    }

    return sums

# Example
s = compute_sums("summer_2025.csv")
print(s)

{'n': 8832, 'sum_y': np.float64(3951935.734779835), 'sum_x1': np.float64(268969.5171726394), 'sum_x2': np.float64(593568.5461345073), 'sum_x1_sq': np.float64(8252499.116144847), 'sum_x2_sq': np.float64(41757813.007711634), 'sum_x1x2': np.float64(17880704.389092457), 'sum_x1y': np.float64(121967614.14421423), 'sum_x2y': np.float64(261047581.01793426)}


In [9]:
import numpy as np

def solve_beta(s):
    A = np.array([
        [s['n'], s['sum_x1'], s['sum_x2']],
        [s['sum_x1'], s['sum_x1_sq'], s['sum_x1x2']],
        [s['sum_x2'], s['sum_x1x2'], s['sum_x2_sq']]
    ])

    B = np.array([
        s['sum_y'],
        s['sum_x1y'],
        s['sum_x2y']
    ])

    beta = np.linalg.solve(A, B)

    return beta

# Example
beta = solve_beta(s)
print("β0, β1, β2 =", beta)

β0, β1, β2 = [-436.04128229   27.92361483    0.49269986]


In [10]:
import pandas as pd
import numpy as np
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

def compare_models(file_name, season):
    df = pd.read_csv(file_name)

    # ------------------------
    # MANUAL MLR (your method)
    # ------------------------
    x1 = df['temp']
    x2 = df['rh']
    y = df['wdLoad']

    n = len(df)

    # Summations
    s = {
        "n": n,
        "sum_y": y.sum(),
        "sum_x1": x1.sum(),
        "sum_x2": x2.sum(),
        "sum_x1_sq": (x1**2).sum(),
        "sum_x2_sq": (x2**2).sum(),
        "sum_x1x2": (x1*x2).sum(),
        "sum_x1y": (x1*y).sum(),
        "sum_x2y": (x2*y).sum()
    }

    # Solve equations
    A = np.array([
        [s['n'], s['sum_x1'], s['sum_x2']],
        [s['sum_x1'], s['sum_x1_sq'], s['sum_x1x2']],
        [s['sum_x2'], s['sum_x1x2'], s['sum_x2_sq']]
    ])

    B = np.array([s['sum_y'], s['sum_x1y'], s['sum_x2y']])
    beta = np.linalg.solve(A, B)

    b0, b1, b2 = beta

    # ------------------------
    # ECONML (causal)
    # ------------------------
    Y = y.values
    T = x1.values        # temp as treatment
    X = x2.values.reshape(-1, 1)  # rh as confounder

    model = LinearDML(
        model_y=RandomForestRegressor(),
        model_t=RandomForestRegressor()
    )

    model.fit(Y, T, X=X)

    causal_effect = model.effect(X).mean()

    # ------------------------
    # PRINT COMPARISON
    # ------------------------
    print(f"\n===== {season} =====")
    print(f"MLR β1 (temp effect): {b1:.4f}")
    print(f"EconML Effect:        {causal_effect:.4f}")
    print(f"MLR β2 (rh effect):   {b2:.4f}")

    # Interpretation
    diff = abs(b1 - causal_effect)

    if diff < 0.1:
        print("✅ Close match → Relationship is likely causal")
    else:
        print("⚠️ Difference → Possible confounding or bias")

# Run for all seasons
compare_models("winter_2025.csv", "Winter")
compare_models("summer_2025.csv", "Summer")
compare_models("monsoon_2025.csv", "Monsoon")
compare_models("autumn_2025.csv", "Autumn")


===== Winter =====
MLR β1 (temp effect): 18.7679
EconML Effect:        20.6318
MLR β2 (rh effect):   -0.1326
⚠️ Difference → Possible confounding or bias

===== Summer =====
MLR β1 (temp effect): 27.9236
EconML Effect:        29.0301
MLR β2 (rh effect):   0.4927
⚠️ Difference → Possible confounding or bias

===== Monsoon =====
MLR β1 (temp effect): 25.1965
EconML Effect:        22.9042
MLR β2 (rh effect):   -0.6180
⚠️ Difference → Possible confounding or bias

===== Autumn =====
MLR β1 (temp effect): 19.2465
EconML Effect:        26.4160
MLR β2 (rh effect):   0.2856
⚠️ Difference → Possible confounding or bias


In [11]:
import pandas as pd
import numpy as np
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

# =========================================
# FUNCTION: Manual MLR (temp, rh, hour)
# =========================================
def manual_mlr_with_hour(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour

    x1 = df['temp']
    x2 = df['rh']
    x3 = df['hour']
    y = df['wdLoad']

    n = len(df)

    # ---- Summations ----
    s = {
        "n": n,
        "sum_y": y.sum(),
        "sum_x1": x1.sum(),
        "sum_x2": x2.sum(),
        "sum_x3": x3.sum(),

        "sum_x1_sq": (x1**2).sum(),
        "sum_x2_sq": (x2**2).sum(),
        "sum_x3_sq": (x3**2).sum(),

        "sum_x1x2": (x1*x2).sum(),
        "sum_x1x3": (x1*x3).sum(),
        "sum_x2x3": (x2*x3).sum(),

        "sum_x1y": (x1*y).sum(),
        "sum_x2y": (x2*y).sum(),
        "sum_x3y": (x3*y).sum()
    }

    # ---- Matrix Form ----
    A = np.array([
        [s['n'], s['sum_x1'], s['sum_x2'], s['sum_x3']],
        [s['sum_x1'], s['sum_x1_sq'], s['sum_x1x2'], s['sum_x1x3']],
        [s['sum_x2'], s['sum_x1x2'], s['sum_x2_sq'], s['sum_x2x3']],
        [s['sum_x3'], s['sum_x1x3'], s['sum_x2x3'], s['sum_x3_sq']]
    ])

    B = np.array([
        s['sum_y'],
        s['sum_x1y'],
        s['sum_x2y'],
        s['sum_x3y']
    ])

    # ---- Solve ----
    beta = np.linalg.solve(A, B)

    return beta  # β0, β1, β2, β3


# =========================================
# FUNCTION: EconML causal effect
# =========================================
def econml_effect(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour

    Y = df['wdLoad'].values
    T = df['temp'].values

    # Confounders
    X = df[['rh', 'hour']].values

    model = LinearDML(
        model_y=RandomForestRegressor(),
        model_t=RandomForestRegressor()
    )

    model.fit(Y, T, X=X)

    effect = model.effect(X).mean()

    return effect


# =========================================
# FUNCTION: Compare BOTH
# =========================================
def compare_season(file_name, season_name):
    df = pd.read_csv(file_name)

    # Manual MLR
    b0, b1, b2, b3 = manual_mlr_with_hour(df)

    # EconML
    causal = econml_effect(df)

    # ---- Print Results ----
    print(f"\n================ {season_name} ================")

    print("\n📘 Mathematical Equation:")
    print(f"wdLoad = {b0:.4f} + ({b1:.4f})·temp + ({b2:.4f})·rh + ({b3:.4f})·hour")

    print("\n📊 Comparison:")
    print(f"MLR Temp Coefficient (β1): {b1:.4f}")
    print(f"EconML Causal Effect:      {causal:.4f}")
    print(f"Hour Effect (β3):          {b3:.4f}")

    # Interpretation
    diff = abs(b1 - causal)

    print("\n🧠 Insight:")
    if diff < 0.1:
        print("✅ MLR matches causal effect → Strong causal relationship")
    else:
        print("⚠️ Difference detected → Possible confounding / bias")


# =========================================
# RUN FOR ALL SEASONS
# =========================================
compare_season("winter_2025.csv", "Winter")
compare_season("summer_2025.csv", "Summer")
compare_season("monsoon_2025.csv", "Monsoon")
compare_season("autumn_2025.csv", "Autumn")


================ Winter ================

📘 Mathematical Equation:
wdLoad = -59.2936 + (17.6090)·temp + (0.0606)·rh + (-3.3121)·hour

📊 Comparison:
MLR Temp Coefficient (β1): 17.6090
EconML Causal Effect:      11.3272
Hour Effect (β3):          -3.3121

🧠 Insight:
⚠️ Difference detected → Possible confounding / bias

================ Summer ================

📘 Mathematical Equation:
wdLoad = -351.3343 + (25.9616)·temp + (0.5334)·rh + (-2.4077)·hour

📊 Comparison:
MLR Temp Coefficient (β1): 25.9616
EconML Causal Effect:      13.2756
Hour Effect (β3):          -2.4077

🧠 Insight:
⚠️ Difference detected → Possible confounding / bias

================ Monsoon ================

📘 Mathematical Equation:
wdLoad = -44.0221 + (21.1970)·temp + (-0.5511)·rh + (-4.8668)·hour

📊 Comparison:
MLR Temp Coefficient (β1): 21.1970
EconML Causal Effect:      11.2527
Hour Effect (β3):          -4.8668

🧠 Insight:
⚠️ Difference detected → Possible confounding / bias

================ Autumn ===============

In [21]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load the winter_2026.csv file
df_winter_2026_hour = pd.read_csv("winter_2026.csv")

# Convert datetime and extract hour
df_winter_2026_hour['datetime'] = pd.to_datetime(df_winter_2026_hour['datetime'])
df_winter_2026_hour['hour'] = df_winter_2026_hour['datetime'].dt.hour

# Coefficients from the new winter equation with 'hour':
# wdLoad = -59.2936 + (17.6090)·temp + (0.0606)·rh + (-3.3121)·hour
intercept_hour = -59.2936
coef_temp_hour = 17.6090
coef_rh_hour = 0.0606
coef_hour = -3.3121

# Predict wdLoad using the new winter equation
df_winter_2026_hour['predicted_wdLoad'] = intercept_hour + \
                                            (coef_temp_hour * df_winter_2026_hour['temp']) + \
                                            (coef_rh_hour * df_winter_2026_hour['rh']) + \
                                            (coef_hour * df_winter_2026_hour['hour'])

# Actual wdLoad values
y_true_hour = df_winter_2026_hour['wdLoad']

# Predicted wdLoad values
y_pred_hour = df_winter_2026_hour['predicted_wdLoad']

# Calculate evaluation metrics

# Mean Absolute Error (MAE)
mae_hour = mean_absolute_error(y_true_hour, y_pred_hour)

# Root Mean Squared Error (RMSE)
rmse_hour = np.sqrt(mean_squared_error(y_true_hour, y_pred_hour))

# Mean Absolute Percentage Error (MAPE)
def mean_absolute_percentage_error(y_true, y_pred):
    # Avoid division by zero
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape_hour = mean_absolute_percentage_error(y_true_hour, y_pred_hour)

print("\n--- Model Performance for Winter 2026 using Winter 2025 Equation (with hour) ---")
print(f"Mean Absolute Error (MAE): {mae_hour:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse_hour:.4f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape_hour:.4f}%")

# Display the first few rows with actual and predicted values
print("\nActual vs. Predicted wdLoad (first 5 rows):")
display(df_winter_2026_hour[['datetime', 'wdLoad', 'predicted_wdLoad']].head())


--- Model Performance for Winter 2026 using Winter 2025 Equation (with hour) ---
Mean Absolute Error (MAE): 62.5748
Root Mean Squared Error (RMSE): 73.2368
Mean Absolute Percentage Error (MAPE): 14.8882%

Actual vs. Predicted wdLoad (first 5 rows):


,datetime,wdLoad,predicted_wdLoad
0,2025-12-01 00:00:00+00:00,298.534462,320.04434
1,2025-12-01 00:15:00+00:00,304.524118,319.18207
2,2025-12-01 00:30:00+00:00,319.374861,318.25920
3,2025-12-01 00:45:00+00:00,329.508837,319.03057
4,2025-12-01 01:00:00+00:00,343.143157,314.79560
